# Numpy 之二：数组的特征（属性）

---

上一篇我们学会了怎么"造"数组，这一篇讲怎么"看"数组 —— 也就是**属性**。

属性（attribute）和函数（function）的区别：

| | 写法 | 要不要括号 |
|---|---|---|
| 属性 | `arr.shape` | **不要**括号 |
| 方法/函数 | `arr.sum()`、`np.sum(arr)` | **要**括号 |

记住这一点：`arr.shape` 是"特征"，`arr.shape()` 是错的。

## 学习目标

1. 记住 7 个核心属性，能一眼说出数组的"身高体重"
2. 弄懂 `itemsize`、`nbytes` 的区别与联系
3. 亲手测一次 list 和 ndarray 的速度差，理解"为什么用 Numpy"


In [ ]:
import numpy as np

arr = np.array([[1, 2, 3],
                [4, 5, 6]])
print(arr)


---
## 一、7 个核心属性

先把它们列出来，再逐个跑：


In [ ]:
# 1. shape —— 形状（每个维度有多长）
print('1. shape    :', arr.shape)      # (2, 3) -> 2 行 3 列

# 2. size —— 元素总个数
print('2. size     :', arr.size)       # 6

# 3. dtype —— 元素的数据类型
print('3. dtype    :', arr.dtype)      # int32

# 4. ndim —— 维度数量（是几维数组）
print('4. ndim     :', arr.ndim)       # 2 -> 二维

# 5. itemsize —— 【单个】元素占多少字节
print('5. itemsize :', arr.itemsize)   # 4 -> 一个 int32 占 4 字节

# 6. nbytes —— 整个数组占多少字节
print('6. nbytes   :', arr.nbytes)     # 24

# 7. data —— 内存缓冲区对象（能看到起始地址）
print('7. data     :', arr.data)       # <memory at 0x...>


### `itemsize` 和 `nbytes` 的关系（常考）

记住这个等式：

```
nbytes = size × itemsize
        = 6   × 4
        = 24
```

**`itemsize` 和 `nbytes` 都是"字节"，不是"个数"**，别和 `size` 混。
它们由 `dtype` 决定：`int32` 是 4 字节，`float64` 就是 8 字节。


In [ ]:
print('验算：size × itemsize =', arr.size * arr.itemsize, '，nbytes =', arr.nbytes)

# 换个 dtype 再感受一次：float64 的 itemsize 翻倍
f = arr.astype(np.float64)
print('\n转成 float64 后：')
print('  dtype    =', f.dtype)
print('  itemsize =', f.itemsize, ' （涨到 8 字节）')
print('  nbytes   =', f.nbytes, ' （随之翻倍）')


> **顺便记住**：`astype()` 是"类型转换"，它会**返回一个新数组**，
> 原来的数组 `arr` 不会变（Numpy 里这种"不改原件"的风格很常见）。


In [ ]:
print('转换后 arr 仍然是：', arr.dtype)


---
## 二、`shape` 是"元组"，不是"参数"

这一点非常容易踩坑。`arr.shape` 得到的是 `(2, 3)` 这样的**元组**，
所以它可以被拆包，也可以拿来喂给 `reshape`：


In [ ]:
rows, cols = arr.shape
print('行数 =', rows, '，列数 =', cols)

# 用 shape 反推：造一个形状完全一样的空数组
blank = np.zeros(arr.shape, dtype=int)
print('照 arr.shape 造的数组：')
print(blank)


### 一维数组的 shape 容易看错


In [ ]:
a1 = np.array([1, 2, 3])
print('一维 arr.shape =', a1.shape, '  <- 注意是 (3,)，不是 (3, 1)')

a2 = np.array([[1], [2], [3]])
print('二维 arr.shape =', a2.shape, '  <- 这才是 (3, 1)')

# 这两者在数值计算里是【完全不同】的，广播时尤其明显（见运算那一篇）
print('\n(3,) 的形状：', a1)
print('\n(3,1) 的形状：')
print(a2)


---
## 三、动手测一次：list 到底慢多少？

只讲"Numpy 快"是空的，我们**实测**。

> **注意**：原笔记里写的是 `range(100000000)`（一亿次），
> 那样光造列表就要几十秒甚至把内存吃满，笔记本里根本跑不动。
> 这里改成**一百万**，既能看出差距，又秒出结果。


In [ ]:
import random
import time

N = 1_000_000        # 一百万。想更直观可以自己改成 1_000_000_0（一千万），但会变慢

# 造一份 Python 原生 list
python_list = [random.random() for _ in range(N)]
# 用同一份数据造 ndarray
ndarray_arr = np.array(python_list)

print('数据量：', N)
print('list 长度：', len(python_list), '，ndarray 长度：', len(ndarray_arr))


In [ ]:
# 计时 1：Python 内置 sum
t1 = time.perf_counter()
s1 = sum(python_list)
t2 = time.perf_counter()
d1 = t2 - t1

# 计时 2：Numpy 的 np.sum
t3 = time.perf_counter()
s2 = np.sum(ndarray_arr)
t4 = time.perf_counter()
d2 = t4 - t3          # 【修正】原笔记写成 t3-t4，会得到负数

print(f'Python sum : {d1:.5f} 秒   结果 = {s1:.4f}')
print(f'Numpy  sum : {d2:.5f} 秒   结果 = {s2:.4f}')
print(f'\nNumpy 快了大约 {d1 / d2:.1f} 倍')


**为什么快这么多？** 两个原因：

1. **存储**：ndarray 是一块**连续内存**，CPU 能一次性把数据搬进缓存；
   list 是一堆散落的对象指针，每次都要"按图索骥"去找。
2. **实现**：`np.sum` 的循环是 C 语言写的，没有 Python 解释器的逐行开销。

> **为什么用 `time.perf_counter()` 而不是 `time.time()`？**
> `perf_counter` 是专门给"测耗时"用的，精度更高、不受系统时钟调整影响。

> **小知识**：原笔记把两种求和的耗时变量搞反了（`d2 = t3 - t4`），
> 结果 Numpy 的耗时是负数。写计时代码时，**永远是"结束时减开始"**。


---
## 四、用分布直方图直观感受随机数

Numpy 生成的是数据，`matplotlib` 负责画出来。
这里只做最简单的演示，**详细的画图在后面的 Matplotlib 篇**。


In [ ]:
import matplotlib.pyplot as plt

# 均匀分布：[-1, 1] 之间，每个值出现的概率差不多 -> 直方图是"平的"
x_uniform = np.random.uniform(-1, 1, 1_000_000)

plt.figure(figsize=(12, 4), dpi=80)
plt.hist(x_uniform, bins=100)
plt.title('uniform(-1, 1) 均匀分布')
plt.show()


In [ ]:
# 正态分布：中间高、两头低 -> 直方图是"钟形"
x_normal = np.random.normal(0, 1, 100_000)

plt.figure(figsize=(12, 4), dpi=80)
plt.hist(x_normal, bins=100)
plt.title('normal(0, 1) 正态分布')
plt.show()


---
## 五、小结：一个"看懂数组"的套路

拿到任何数组，先问四句话：

1. **几维？** `arr.ndim`
2. **多长？** `arr.shape`
3. **多少个？** `arr.size`
4. **什么类型、占多大？** `arr.dtype` / `arr.itemsize` / `arr.nbytes`

这四个问题答完，你就知道手里这坨数据"长什么样、有多重"了。


---
## 六、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `arr.shape()` | `TypeError: 'tuple' object is not callable` | 属性**不加括号**：`arr.shape` |
| 把 `size` 当字节数 | 数字对不上 | `size` 是**元素个数**，字节数看 `nbytes` |
| 把 `(3,)` 当成 `(3, 1)` | 广播结果不符合预期 | 一维就是 `(3,)`；要列向量用 `[[1],[2],[3]]` |
| `astype()` 以为改了自己 | 原数组没变 | `astype` **返回新数组**，要接收返回值 |
| 计时写成 `t_end - t_start` 的反面 | 耗时是负数 | 结束时间 **减** 开始时间 |
| 循环里用 `for i in range(1亿)` | 卡死 / 内存爆 | 演示用 1e6 量级即可 |

---

## 七、动手练习

**练习 1**：创建一个 `(3, 4, 5)` 的三维数组，打印出它的 `ndim`、`shape`、`size`、`nbytes`。

**练习 2**：有一个 `int8` 数组，算算它的 `nbytes` 和 `size` 是什么关系，
再换成 `float32` 看倍数变化。

**练习 3**：写出判断"两个数组形状相同"的代码（提示：比较 `shape`）。

**练习 4**：把 `np.arange(12)` 分别 reshape 成 `(3, 4)` 和 `(4, 3)`，
用 `shape` 属性验证结果，并说说数据"填"的顺序是怎样的。

---

**答案思路（先自己做）**

```python
# 1
a = np.zeros((3, 4, 5))
print(a.ndim, a.shape, a.size, a.nbytes)

# 2
b = np.zeros(10, dtype='int8')
print(b.nbytes, b.size, b.itemsize)      # 10 10 1

# 3
print(np.array([1,2,3]).shape == np.array([4,5,6]).shape)

# 4
print(np.arange(12).reshape(3, 4).shape)
print(np.arange(12).reshape(4, 3).shape)   # 都是 12 个元素，形状不同
```
